# INF610 — Benchmark de posicionamento: YOTO × YOTT × Simulated Annealing

Aqui tem todo o código do trabalho: leitura dos grafos, construção da grade, os três
algoritmos, os experimentos, as figuras e os vídeos.

Para rodar, é só clicar em **Executar tudo**. Os grafos são baixados do GitHub
automaticamente; não é preciso enviar nenhum arquivo.<br>

O último bloco está comentado pois ele baixa um `.zip` com os resultados. Caso queira, basta descomentar ele e executar

| Etapa | O que faz |
|---|---|
| 0 | Configuração |
| 1 | Bibliotecas |
| 2 | Ler os grafos (nós e arestas) |
| 3 | Construir a grade |
| 4 | Medir a qualidade de um posicionamento |
| 5 | Travessia zig-zag |
| 6 | YOTO |
| 7 | YOTT |
| 8 | Simulated Annealing |
| 9 | Multi-start (×100) |
| 10 | Exemplo pequeno |
| 11 | Vídeos |
| 12 | Experimentos E1 a E5 |
| 13 | Figuras e tabelas |
| 14 | Mostrar e baixar os resultados |

## 0. Configuração

In [ ]:
REPO_URL = "https://github.com/Danntav/INF610-DataStructures-Algorithms.git"  # onde estão os grafos
PASTA_SAIDA = "resultados"
GRAFO_EXEMPLO = "optimal_6x6_s0"

## 1. Bibliotecas
Só bibliotecas que o Colab já traz. A última linha registra a máquina usada (vai para o relatório).

In [ ]:
import math
import os
import platform
import random
import re
import subprocess
import time
import tracemalloc
import zipfile
from collections import Counter, deque
from dataclasses import dataclass, field
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib import animation
from IPython.display import Image, Markdown, Video, display

SAIDA, FIGURAS, VIDEOS = Path(PASTA_SAIDA), Path(PASTA_SAIDA) / "figuras", Path(PASTA_SAIDA) / "videos"
FIGURAS.mkdir(parents=True, exist_ok=True)
VIDEOS.mkdir(parents=True, exist_ok=True)

ambiente = f"Python {platform.python_version()} | {platform.platform()} | {os.cpu_count()} núcleos"
if Path("/proc/cpuinfo").exists():   # no Colab: modelo da CPU
    ambiente += " | " + next(l.split(":")[1].strip() for l in open("/proc/cpuinfo") if l.startswith("model name"))
(SAIDA / "ambiente.txt").write_text(ambiente, encoding="utf-8")
print(ambiente)

## 2. Ler os grafos
Os grafos estão no GitHub, em três pastas:
- `data/real/`: 23 DFGs reais do LESC/UFV;
- `data/synthetic/`: 35 grafos sintéticos em camadas (64 a 4096 nós);
- `data/optimal/`: 50 grafos com posicionamento ótimo conhecido (WL ótimo = |E|).

In [ ]:
if Path("data/real").exists():
    PASTA_DADOS = Path("data")
else:
    if not Path("repositorio").exists():
        subprocess.run(["git", "clone", "--depth", "1", "--filter=blob:none", "--sparse", REPO_URL, "repositorio"],
                       check=True)
        subprocess.run(["git", "-C", "repositorio", "sparse-checkout", "set", "data"], check=True)
    PASTA_DADOS = Path("repositorio/data")

### Separar nós e arestas e montar o grafo
Num arquivo `.dot`, `a;` declara um nó e `a -> b;` declara uma aresta (pode vir em cadeia: `a -> b -> c;`).
Comentários (`//`) e atributos entre colchetes (`[opcode=add]`) são descartados.

Os nós são renumerados de 0 a |V|−1, e cada nó ganha três **listas de adjacência**: `succ` (para onde suas
arestas vão), `pred` (de onde vêm) e `adj` (todos os vizinhos, sem direção, porque o custo não depende da
direção). Arestas repetidas e laços (v → v) são descartados.

In [ ]:
def ler_dot(texto):
    """Devolve (nós na ordem em que aparecem, arestas como pares de nomes)."""
    nos, arestas = {}, []                              # dict = conjunto que guarda a ordem de inserção
    for linha in texto.splitlines():
        linha = re.sub(r"\[.*?\]", "", linha.split("//")[0])
        for comando in linha.split(";"):
            partes = [p.strip() for p in comando.split("->")]
            if not all(re.fullmatch(r"[\w.]+", p) for p in partes):
                continue                               # cabeçalho "digraph X {", "}" ou linha vazia
            for p in partes:
                nos.setdefault(p)
            arestas += zip(partes, partes[1:])
    return list(nos), arestas


class Grafo:
    def __init__(self, nome, texto_dot):
        nomes, arestas_nomeadas = ler_dot(texto_dot)
        indice = {x: i for i, x in enumerate(nomes)}
        self.nome, self.nomes = nome, nomes
        self.arestas = list(dict.fromkeys((indice[a], indice[b]) for a, b in arestas_nomeadas if a != b))
        self.n_nos, self.n_arestas = len(nomes), len(self.arestas)
        self.succ = [[] for _ in range(self.n_nos)]
        self.pred = [[] for _ in range(self.n_nos)]
        for u, v in self.arestas:
            self.succ[u].append(v)
            self.pred[v].append(u)
        self.adj = [sorted(set(self.succ[v] + self.pred[v])) for v in range(self.n_nos)]

    def saidas(self):
        """Nós sem sucessores: é por eles que a travessia começa."""
        return [v for v in range(self.n_nos) if not self.succ[v]]


def carregar_pasta(pasta):
    """Todos os .dot de uma pasta, em ordem crescente de nº de nós."""
    grafos = [Grafo(arq.stem, arq.read_text(encoding="utf-8")) for arq in Path(pasta).glob("*.dot")]
    return sorted(grafos, key=lambda g: (g.n_nos, g.nome))


REAIS = carregar_pasta(PASTA_DADOS / "real")
SINTETICOS = {g.nome: g for g in carregar_pasta(PASTA_DADOS / "synthetic")}
OTIMOS = {g.nome: g for g in carregar_pasta(PASTA_DADOS / "optimal")}
TODOS = {g.nome: g for g in [*REAIS, *SINTETICOS.values(), *OTIMOS.values()]}
print(f"{len(REAIS)} reais, {len(SINTETICOS)} sintéticos, {len(OTIMOS)} de ótimo conhecido")
display(pd.DataFrame([(g.nome, g.n_nos, g.n_arestas, max(map(len, g.adj))) for g in REAIS],
                     columns=["grafo", "nós", "arestas", "grau máximo"]))

## 3. Construir a grade
A grade tem R linhas e C colunas. Cada célula recebe um número `c = linha·C + coluna`.
- `ocupacao[c]`: qual nó está na célula c (ou `LIVRE`);
- `vizinhas[c]`: as até 4 células vizinhas de c (norte, sul, leste, oeste), calculadas uma vez só;
- distância entre células = **distância de Manhattan** (|Δlinha| + |Δcoluna|);
- `livre_mais_proxima(c)`: **BFS** a partir de c até achar uma célula livre (usada pelo YOTO e pelo YOTT
  quando nenhuma vizinha está livre).

O tamanho é sempre a **grade quadrada mínima**: lado = ⌈√|V|⌉.

In [ ]:
LIVRE = -1


class Grade:
    def __init__(self, linhas, colunas):
        self.linhas, self.colunas = linhas, colunas
        self.tamanho = linhas * colunas
        self.ocupacao = [LIVRE] * self.tamanho
        self.lc = [(c // colunas, c % colunas) for c in range(self.tamanho)]  # (linha, coluna) de cada célula
        self.vizinhas = []
        for c in range(self.tamanho):
            r, k = self.lc[c]
            viz = []
            if r > 0:
                viz.append(c - colunas)       # norte
            if r < linhas - 1:
                viz.append(c + colunas)       # sul
            if k < colunas - 1:
                viz.append(c + 1)             # leste
            if k > 0:
                viz.append(c - 1)             # oeste
            self.vizinhas.append(viz)

    @classmethod
    def para_grafo(cls, n_nos):
        lado = max(1, math.ceil(math.sqrt(n_nos)))
        return cls(lado, lado)

    def vazia(self):
        """Cópia sem nós. Só a ocupação é nova; `lc` e `vizinhas` (a geometria) são compartilhadas,
        porque não mudam e fazem parte da entrada, não da memória extra do algoritmo."""
        nova = object.__new__(Grade)
        nova.__dict__.update(self.__dict__)
        nova.ocupacao = [LIVRE] * self.tamanho
        return nova

    def dist(self, a, b):
        ra, ka = self.lc[a]
        rb, kb = self.lc[b]
        return abs(ra - rb) + abs(ka - kb)

    def colocar(self, no, c):
        assert self.ocupacao[c] == LIVRE, f"célula {c} já ocupada"
        self.ocupacao[c] = no

    def vizinhas_livres(self, c):
        return [x for x in self.vizinhas[c] if self.ocupacao[x] == LIVRE]

    def anel(self, c, d):
        """Células exatamente a distância d de c."""
        r0, k0 = self.lc[c]
        saida = []
        for dr in range(-d, d + 1):
            resto = d - abs(dr)
            r = r0 + dr
            if not 0 <= r < self.linhas:
                continue
            for dk in {resto, -resto}:
                k = k0 + dk
                if 0 <= k < self.colunas:
                    saida.append(r * self.colunas + k)
        return saida

    def livre_mais_proxima(self, c):
        """BFS: numa grade com 4 vizinhos, a BFS visita as células em ordem crescente de distância,
        então a primeira livre encontrada é a mais próxima."""
        if self.ocupacao[c] == LIVRE:
            return c
        visitada = bytearray(self.tamanho)
        visitada[c] = 1
        fila = deque([c])
        while fila:
            x = fila.popleft()
            for y in self.vizinhas[x]:
                if not visitada[y]:
                    if self.ocupacao[y] == LIVRE:
                        return y
                    visitada[y] = 1
                    fila.append(y)
        raise RuntimeError("grade cheia")

## 4. Medir a qualidade de um posicionamento
Um posicionamento é uma lista `pos`, em que `pos[v]` é a célula do nó v.
- **WL** (wirelength): soma das distâncias de Manhattan de todas as arestas. Menor é melhor.
- **LB** (limitante inferior): nenhum posicionamento tem WL menor que isso. Toda aresta mede pelo menos 1, e um
  nó com mais de 4 vizinhos é obrigado a ter arestas mais longas (só há 4 células a distância 1, 8 a distância 2…).
- **validar**: confere que todo nó tem uma célula e que nenhuma célula tem dois nós.

In [ ]:
def comprimentos(grafo, grade, pos):
    return [grade.dist(pos[u], pos[v]) for u, v in grafo.arestas]


def wirelength(grafo, grade, pos):
    return sum(comprimentos(grafo, grade, pos))


def limitante_inferior(grafo):
    excesso = 0
    for v in range(grafo.n_nos):
        grau = len(grafo.adj[v])
        custo, raio, faltam = 0, 1, grau
        while faltam > 0:                  # preenche os anéis de raio 1, 2, 3... (4·raio células cada)
            cabem = min(faltam, 4 * raio)
            custo += cabem * raio
            faltam -= cabem
            raio += 1
        excesso += custo - grau            # quanto passa de "todas as arestas com comprimento 1"
    return grafo.n_arestas + math.ceil(excesso / 2)   # cada aresta foi contada pelas duas pontas


def validar(grafo, grade, pos):
    assert len(pos) == grafo.n_nos, "posicionamento incompleto"
    assert all(0 <= c < grade.tamanho for c in pos), "célula fora da grade"
    assert len(set(pos)) == len(pos), "dois nós na mesma célula"


def metricas(grafo, grade, pos):
    comp = comprimentos(grafo, grade, pos)
    wl, lb = sum(comp), limitante_inferior(grafo)
    return {"wl": wl, "lb": lb, "gap_lb": wl / lb, "wl_max": max(comp),
            "pct_arestas_otimas": 100 * Counter(comp)[1] / len(comp)}

## 5. Travessia zig-zag
Define a **ordem** em que o YOTO e o YOTT posicionam os nós. É uma busca em profundidade (com pilha) que
percorre **cada aresta uma vez**, começando pelas saídas do grafo. A DFS comum sempre prefere seguir para os
predecessores; o zig-zag **alterna**: se chegou a um nó subindo (por um predecessor), tenta em seguida descer
por um sucessor, e vice-versa. Só segue na mesma direção quando não há aresta na outra.

Cada passo é `(a, b, novo)`:
- `novo = True`: b foi visitado agora, vindo de a → b será posicionado perto de a;
- `novo = False`: b já tinha sido visitado → é uma **reconvergência** (usada pelo YOTT);
- `a = None`: b é a raiz (primeiro nó de um componente).

Estruturas: listas de arestas ainda não percorridas (`resta_pred`, `resta_succ`), vetor `visitado`,
**pilha** da busca em profundidade.

In [ ]:
PARA_PRED, PARA_SUCC = 0, 1


def travessia_zigzag(grafo, rng):
    n = grafo.n_nos
    resta_pred = [list(p) for p in grafo.pred]
    resta_succ = [list(s) for s in grafo.succ]
    for lista in resta_pred + resta_succ:          # cada instância percorre numa ordem diferente
        rng.shuffle(lista)
    visitado = [False] * n
    passos = []

    raizes = grafo.saidas()
    rng.shuffle(raizes)
    sao_raiz = set(raizes)
    raizes += [v for v in range(n) if v not in sao_raiz]

    for raiz in raizes:
        if visitado[raiz]:
            continue
        visitado[raiz] = True
        passos.append((None, raiz, True))
        pilha = [(raiz, PARA_PRED)]
        while pilha:
            a, direcao = pilha[-1]
            primeira, segunda = (PARA_SUCC, PARA_PRED) if direcao == PARA_PRED else (PARA_PRED, PARA_SUCC)
            if (resta_succ if primeira == PARA_SUCC else resta_pred)[a]:
                nova = primeira                    # dá para trocar de direção: troca
            elif (resta_succ if segunda == PARA_SUCC else resta_pred)[a]:
                nova = segunda
            else:
                pilha.pop()                        # a não tem mais arestas: volta
                continue
            if nova == PARA_SUCC:
                b = resta_succ[a].pop()
                resta_pred[b].remove(a)
            else:
                b = resta_pred[a].pop()
                resta_succ[b].remove(a)
            if visitado[b]:
                passos.append((a, b, False))       # reconvergência
            else:
                visitado[b] = True
                passos.append((a, b, True))
                pilha.append((b, nova))
    return passos

### Resultado de um algoritmo
Os três algoritmos devolvem o mesmo tipo de resultado. `contadores` guarda informações para a análise
(chamadas de BFS do YOTO/YOTT, temperaturas do SA).

In [ ]:
@dataclass
class Resultado:
    pos: list
    wl: int
    tempo: float
    contadores: dict = field(default_factory=dict)
    curva: list = field(default_factory=list)   # (tempo, melhor WL) — usada no multi-start


def celula_inicial(grade, ultima, rng):
    """Primeiro nó: célula aleatória. Componentes seguintes: perto do último nó colocado."""
    inicio = rng.randrange(grade.tamanho) if ultima is None else ultima
    return grade.livre_mais_proxima(inicio)

## 6. YOTO — You Only Traverse Once
Uma travessia define a ordem. Para cada passo (a → b), o nó b vai para uma **vizinha livre de a, sorteada**.
Se as 4 vizinhas estiverem ocupadas, uma **BFS** acha a célula livre mais próxima de a.
As reconvergências são ignoradas.

`registro` (opcional) guarda cada colocação, para o vídeo.

In [ ]:
def yoto(grafo, grade_modelo, semente, registro=None):
    rng = random.Random(semente)
    inicio = time.perf_counter()
    grade = grade_modelo.vazia()
    passos = travessia_zigzag(grafo, rng)
    sequencia = [p for p in passos if p[2]]            # só os passos que colocam um nó novo
    pos = [-1] * grafo.n_nos
    chamadas_bfs, ultima = 0, None
    for a, b, _ in sequencia:
        if a is None:
            c = celula_inicial(grade, ultima, rng)
        else:
            livres = grade.vizinhas_livres(pos[a])
            if livres:
                c = rng.choice(livres)
            else:
                chamadas_bfs += 1
                c = grade.livre_mais_proxima(pos[a])
        grade.colocar(b, c)
        pos[b] = c
        ultima = c
        if registro is not None:
            registro.append((b, c, False))
    tempo = time.perf_counter() - inicio
    return Resultado(pos, wirelength(grafo, grade, pos), tempo, {"bfs": chamadas_bfs})

## 7. YOTT — You Only Traverse Twice
**1ª passada (anotações).** Para cada reconvergência (a aresta liga `cedo`, visitado antes, a `tarde`), anota
que `tarde` deve ficar a distância 1 de `cedo`, o pai de `tarde` a distância 2, o avô a distância 3
(até `max_dist` = 3). Só são anotados nós visitados depois de `cedo`. Quando o ciclo se fecha pela própria
árvore, a distância-alvo é ajustada para a metade do ciclo.

**2ª passada (posicionamento guiado).** Mesma ordem do YOTO, mas as candidatas (vizinhas livres do pai, ou o
anel livre mais próximo achado pela BFS) são comparadas por prioridade:
1. penalidade das anotações (quanto cada anotação é desrespeitada);
2. look-ahead (anotações que ficariam impossíveis de cumprir);
3. degree matching (nº de vizinhas livres da célula × nº de vizinhos do nó que faltam posicionar);
4. desempate aleatório.

Estruturas extras: vetores `ordem` e `pai` (|V| cada), lista de anotações por nó, vetor `faltam` (|V|).

In [ ]:
def anotar(grafo, passos, max_dist=3):
    n = grafo.n_nos
    ordem = [0] * n            # ordem[v] = posição de v na visita
    pai = [-1] * n             # pai[v] = nó de onde a travessia chegou em v
    k = 0
    for a, b, novo in passos:
        if novo:
            ordem[b] = k
            k += 1
            if a is not None:
                pai[b] = a

    anotacoes = [[] for _ in range(n)]     # anotacoes[v] = [(âncora, distância desejada), ...]
    for a, b, novo in passos:
        if novo:
            continue
        cedo, tarde = (a, b) if ordem[a] < ordem[b] else (b, a)
        # ancestral comum mais próximo (procura até 2·max_dist níveis acima)
        anc_cedo = {}
        u, d = cedo, 0
        while u != -1 and d <= 2 * max_dist:
            anc_cedo[u] = d
            u, d = pai[u], d + 1
        sobe_tarde = sobe_cedo = None
        u, d = tarde, 0
        while u != -1 and d <= 2 * max_dist:
            if u in anc_cedo:
                sobe_tarde, sobe_cedo = d, anc_cedo[u]
                break
            u, d = pai[u], d + 1
        # anota tarde, pai(tarde), avô(tarde)... enquanto forem visitados depois de cedo
        u, j = tarde, 0
        while u != -1 and j < max_dist and ordem[u] > ordem[cedo]:
            alvo = j + 1
            if sobe_tarde is not None and j <= sobe_tarde:
                alvo = min(alvo, (sobe_tarde - j) + sobe_cedo)   # fechar o ciclo pela árvore
            anotacoes[u].append((cedo, alvo))
            u, j = pai[u], j + 1
    return anotacoes


def yott(grafo, grade_modelo, semente, registro=None):
    rng = random.Random(semente)
    inicio = time.perf_counter()
    grade = grade_modelo.vazia()
    ocup, viz, dist = grade.ocupacao, grade.vizinhas, grade.dist

    # 1ª passada
    passos = travessia_zigzag(grafo, rng)
    anotacoes = anotar(grafo, passos)

    # 2ª passada
    sequencia = [p for p in passos if p[2]]
    pos = [-1] * grafo.n_nos
    faltam = [len(a) for a in grafo.adj]       # vizinhos de cada nó ainda sem posição
    chamadas_bfs, ultima = 0, None
    for a, b, _ in sequencia:
        if a is None:
            c = celula_inicial(grade, ultima, rng)
        else:
            pa = pos[a]
            candidatas = [x for x in viz[pa] if ocup[x] == LIVRE]
            if not candidatas:
                chamadas_bfs += 1
                raio = dist(pa, grade.livre_mais_proxima(pa))
                candidatas = [x for x in grade.anel(pa, raio) if ocup[x] == LIVRE]
            if len(candidatas) == 1:
                c = candidatas[0]
            else:
                alvos = [(pos[x], d) for x, d in anotacoes[b]]
                precisa = faltam[b]
                melhor, c = None, None
                for x in candidatas:
                    penalidade = sum(max(0, dist(x, px) - d) for px, d in alvos)
                    falhas = sum(1 for px, d in alvos
                                 if d >= 2 and not any(ocup[y] == LIVRE and dist(y, px) <= d - 1 for y in viz[x]))
                    livres = sum(1 for y in viz[x] if ocup[y] == LIVRE)
                    casamento = (max(0, precisa - livres), max(0, livres - precisa))
                    chave = (penalidade, falhas, casamento, rng.random())
                    if melhor is None or chave < melhor:
                        melhor, c = chave, x
        grade.colocar(b, c)
        pos[b] = c
        ultima = c
        for w in grafo.adj[b]:
            faltam[w] -= 1
        if registro is not None:
            registro.append((b, c, bool(anotacoes[b])))
    tempo = time.perf_counter() - inicio
    return Resultado(pos, wirelength(grafo, grade, pos), tempo, {"bfs": chamadas_bfs})

## 8. Simulated Annealing
Parte de um posicionamento **completo e aleatório** e o melhora com pequenas mudanças:
1. sorteia um nó v e uma célula c perto dele (a distância máxima é a "janela", que encolhe com o tempo);
2. se c está livre, move v para c; se está ocupada por w, troca v e w;
3. calcula a variação ΔWL **só nas arestas de v e w** (incremental, O(grau));
4. regra de Metropolis: aceita se ΔWL ≤ 0; se ΔWL > 0, aceita com probabilidade e^(−ΔWL/T).

A temperatura começa em T0 (calibrada para aceitar ~90% das pioras) e é multiplicada por um fator α que
depende da taxa de aceitação (resfriamento adaptativo, estilo VPR). São `k`·|V| tentativas por temperatura
(k = 10). Para quando T < 0,005·WL/|E| e faz uma última passada só com melhorias.

Estruturas: vetores `pos` (célula de cada nó) e `ocup` (nó de cada célula), linha e coluna de cada célula.
`registro` (opcional) guarda o estado ao fim de cada temperatura, para o vídeo.

In [ ]:
def simulated_annealing(grafo, grade, semente, k=10.0, registro=None):
    rng = random.Random(semente)
    inicio = time.perf_counter()
    n, M = grafo.n_nos, grade.tamanho
    linhas, colunas = grade.linhas, grade.colunas
    L = [c // colunas for c in range(M)]       # linha de cada célula
    C = [c % colunas for c in range(M)]        # coluna de cada célula
    adj = grafo.adj

    # estado inicial aleatório
    celulas = list(range(M))
    rng.shuffle(celulas)
    pos = celulas[:n]
    ocup = [LIVRE] * M
    for v, c in enumerate(pos):
        ocup[c] = v

    def custo_no(v, cv):
        """Soma dos comprimentos das arestas de v, se v estivesse na célula cv."""
        lv, kv = L[cv], C[cv]
        return sum(abs(lv - L[pos[w]]) + abs(kv - C[pos[w]]) for w in adj[v])

    custo = sum(abs(L[pos[u]] - L[pos[v]]) + abs(C[pos[u]] - C[pos[v]]) for u, v in grafo.arestas)

    def propor(janela):
        v = rng.randrange(n)
        cv = pos[v]
        while True:
            r = min(linhas - 1, max(0, L[cv] + rng.randint(-janela, janela)))
            q = min(colunas - 1, max(0, C[cv] + rng.randint(-janela, janela)))
            c = r * colunas + q
            if c != cv:
                return v, c

    def delta(v, c):
        cv, w = pos[v], ocup[c]
        antes = custo_no(v, cv)
        if w == LIVRE:
            pos[v] = c
            depois = custo_no(v, c)
            pos[v] = cv
            return depois - antes
        antes += custo_no(w, c)
        pos[v], pos[w] = c, cv
        depois = custo_no(v, c) + custo_no(w, cv)
        pos[v], pos[w] = cv, c
        return depois - antes

    def aplicar(v, c):
        cv, w = pos[v], ocup[c]
        pos[v], ocup[c] = c, v
        ocup[cv] = w
        if w != LIVRE:
            pos[w] = cv

    # temperatura inicial: ~90% das pioras aceitas
    janela_max = max(linhas, colunas)
    pioras = [d for d in (delta(*propor(janela_max)) for _ in range(max(20, n))) if d > 0]
    T = -(sum(pioras) / len(pioras)) / math.log(0.9) if pioras else 1.0

    por_temp = max(1, int(k * n))
    janela = janela_max
    melhor_custo, melhor_pos = custo, pos[:]
    tentativas = n_temps = 0
    if registro is not None:
        registro.append((0, 0, T, custo, pos[:]))
    while True:
        aceitas = 0
        for _ in range(por_temp):
            v, c = propor(janela)
            d = delta(v, c)
            if d <= 0 or rng.random() < math.exp(-d / T):
                aplicar(v, c)
                custo += d
                aceitas += 1
        tentativas += por_temp
        n_temps += 1
        taxa = aceitas / por_temp
        if custo < melhor_custo:
            melhor_custo, melhor_pos = custo, pos[:]
        if registro is not None:
            registro.append((n_temps, tentativas, T, custo, pos[:]))
        # resfriamento adaptativo: esfria rápido se aceita quase tudo, devagar na faixa útil
        alfa = 0.5 if taxa > 0.96 else 0.9 if taxa > 0.8 else 0.95 if taxa > 0.15 else 0.8
        janela = max(1, min(janela_max, round(janela * (1 - 0.44 + taxa))))
        T *= alfa
        if T < 0.005 * custo / max(1, grafo.n_arestas):
            break

    # passada final só com melhorias (T = 0)
    for _ in range(por_temp):
        v, c = propor(1)
        d = delta(v, c)
        if d < 0:
            aplicar(v, c)
            custo += d
    tentativas += por_temp
    if custo < melhor_custo:
        melhor_custo, melhor_pos = custo, pos[:]
    if registro is not None:
        registro.append((n_temps, tentativas, 0.0, melhor_custo, melhor_pos[:]))

    tempo = time.perf_counter() - inicio
    return Resultado(melhor_pos, melhor_custo, tempo, {"temperaturas": n_temps, "tentativas": tentativas})

## 9. Multi-start (×100)
YOTO e YOTT são rápidos e aleatórios: rodar várias instâncias independentes e ficar com a melhor costuma dar um
resultado bem melhor (é o que o artigo do YOTT faz). Para quando completa `instancias` ou quando o tempo passa de
`orcamento` segundos (usado no E3). `curva` guarda (tempo, melhor WL) após cada instância.

In [ ]:
def multi_start(algoritmo, grafo, grade, semente, instancias=100, orcamento=None):
    if orcamento is not None:
        instancias = 10 ** 9
    sorteio = random.Random(semente)            # gera uma semente diferente para cada instância
    inicio = time.perf_counter()
    melhor, curva, feitas = None, [], 0
    while feitas < instancias:
        r = algoritmo(grafo, grade, sorteio.getrandbits(32))
        feitas += 1
        if melhor is None or r.wl < melhor.wl:
            melhor = r
        agora = time.perf_counter() - inicio
        curva.append((agora, melhor.wl))
        if orcamento is not None and agora >= orcamento:
            break
    melhor.tempo = time.perf_counter() - inicio
    melhor.contadores = {**melhor.contadores, "instancias": feitas}
    melhor.curva = curva
    return melhor


# os cinco algoritmos comparados (sempre com esta configuração)
ALGORITMOS = {
    "YOTO":      lambda g, gr, s: multi_start(yoto, g, gr, s, instancias=1),
    "YOTT":      lambda g, gr, s: multi_start(yott, g, gr, s, instancias=1),
    "YOTO-x100": lambda g, gr, s: multi_start(yoto, g, gr, s, instancias=100),
    "YOTT-x100": lambda g, gr, s: multi_start(yott, g, gr, s, instancias=100),
    "SA":        lambda g, gr, s: simulated_annealing(g, gr, s),
}

## 10. Exemplo pequeno
Os três algoritmos no mesmo grafo pequeno, lado a lado. Serve para conferir visualmente que tudo funciona.
Verde = aresta de comprimento 1 (ótima); vermelho = aresta mais longa.

In [ ]:
CORES = {"YOTO": "#2a78d6", "YOTT": "#eb6834", "YOTO-x100": "#1baf7a", "YOTT-x100": "#eda100", "SA": "#e87ba4"}
MARCADORES = {"YOTO": "o", "YOTO-x100": "o", "YOTT": "s", "YOTT-x100": "s", "SA": "D"}
ROTULOS = {"YOTO": "YOTO ×1", "YOTT": "YOTT ×1", "YOTO-x100": "YOTO ×100", "YOTT-x100": "YOTT ×100", "SA": "SA"}
ORDEM = list(ROTULOS)
plt.rcParams.update({"axes.grid": True, "grid.color": "#e1e0d9", "axes.spines.top": False,
                     "axes.spines.right": False, "legend.frameon": False, "font.size": 10})


def desenhar(ax, grafo, grade, pos, titulo, destaque=None, cor_no="#52514e"):
    """Desenha a grade, as arestas já posicionadas e os nós. `pos[v] = -1` = nó ainda sem célula."""
    ax.clear()
    ax.set_xlim(-0.5, grade.colunas - 0.5)
    ax.set_ylim(grade.linhas - 0.5, -0.5)
    ax.set_aspect("equal")
    ax.set_xticks(np.arange(-0.5, grade.colunas, 1))
    ax.set_yticks(np.arange(-0.5, grade.linhas, 1))
    ax.tick_params(length=0, labelbottom=False, labelleft=False)
    ax.grid(True, color="#e1e0d9", lw=0.8)
    for u, v in grafo.arestas:
        if pos[u] >= 0 and pos[v] >= 0:
            (ru, ku), (rv, kv) = grade.lc[pos[u]], grade.lc[pos[v]]
            curta = grade.dist(pos[u], pos[v]) == 1
            ax.plot([ku, kv], [ru, rv], color="#1baf7a" if curta else "#d03b3b",
                    lw=1.6 if curta else 1.0, alpha=0.9 if curta else 0.55, zorder=1)
    pontos = [grade.lc[c] for c in pos if c >= 0]
    if pontos:
        ax.scatter([k for _, k in pontos], [r for r, _ in pontos], s=60, color=cor_no, zorder=2)
    if destaque is not None:
        r, k = grade.lc[destaque]
        ax.scatter([k], [r], s=170, facecolor="none", edgecolor="#0b0b0b", lw=2.0, zorder=3)
    ax.set_title(titulo, loc="left", fontsize=10)


g_ex = TODOS[GRAFO_EXEMPLO]
gr_ex = Grade.para_grafo(g_ex.n_nos)
print(f"{g_ex.nome}: {g_ex.n_nos} nós, {g_ex.n_arestas} arestas, grade {gr_ex.linhas}×{gr_ex.colunas}, "
      f"LB = {limitante_inferior(g_ex)}")

### O grafo de exemplo, antes de posicionar
Desenho "livre" do grafo (sem grade), só para ver a sua forma. As posições vêm de um layout de força
(Kamada-Kawai, da biblioteca networkx): nós ligados ficam perto, e o desenho sai compacto.

In [ ]:
import networkx as nx


def desenhar_grafo(ax, grafo):
    G = nx.DiGraph()
    G.add_nodes_from(range(grafo.n_nos))
    G.add_edges_from(grafo.arestas)
    sem_direcao = G.to_undirected()
    xy = nx.kamada_kawai_layout(sem_direcao, pos=nx.spectral_layout(sem_direcao))
    nx.draw_networkx(G, xy, ax=ax, labels={v: grafo.nomes[v] for v in G}, node_color="#d9d8d2",
                     edgecolors="#52514e", node_size=420, font_size=8, arrowsize=12, min_target_margin=10)
    ax.set_title(f"{grafo.nome}: {grafo.n_nos} nós, {grafo.n_arestas} arestas", loc="left", fontsize=10)
    ax.axis("off")


fig, ax = plt.subplots(figsize=(8, 6))
desenhar_grafo(ax, g_ex)
plt.tight_layout()
plt.savefig(FIGURAS / "grafo_exemplo.png", dpi=150)
plt.show()

### Os três posicionamentos na grade

In [ ]:
fig, axs = plt.subplots(1, 3, figsize=(13, 4.6))
for ax, (nome, alg) in zip(axs, (("YOTO", yoto), ("YOTT", yott), ("SA", simulated_annealing))):
    r = alg(g_ex, gr_ex, 0)
    validar(g_ex, gr_ex, r.pos)
    desenhar(ax, g_ex, gr_ex, r.pos, f"{nome}: WL = {r.wl}", cor_no=CORES[nome])
plt.tight_layout()
plt.savefig(FIGURAS / "exemplo_posicionamentos.png", dpi=150)
plt.show()

## 11. Vídeos
Um vídeo por algoritmo, no grafo do exemplo:
- **YOTO e YOTT**: um nó por quadro, na ordem da travessia (o círculo preto marca o nó que acabou de entrar;
  no YOTT, nós que receberam anotação aparecem em roxo);
- **SA**: um quadro por temperatura, com o nº de tentativas, a temperatura e o WL no título.

Gera `.mp4` (no Colab) ou `.gif` (se não houver ffmpeg).

In [ ]:
def salvar_animacao(fig, atualizar, n_quadros, nome, fps):
    anim = animation.FuncAnimation(fig, atualizar, frames=n_quadros, interval=1000 / fps)
    if animation.writers.is_available("ffmpeg"):
        caminho = VIDEOS / f"{nome}.mp4"
        anim.save(caminho, writer=animation.FFMpegWriter(fps=fps, bitrate=1800), dpi=120)
    else:
        caminho = VIDEOS / f"{nome}.gif"
        anim.save(caminho, writer=animation.PillowWriter(fps=fps), dpi=90)
    plt.close(fig)
    return caminho


def video_construtivo(nome, algoritmo, grafo, grade, semente=0, fps=4):
    registro = []
    r = algoritmo(grafo, grade, semente, registro=registro)
    fig, ax = plt.subplots(figsize=(5.5, 5.8))
    pausa = 2 * fps                                  # segura o quadro final por 2 s

    def atualizar(i):
        i = min(i, len(registro) - 1)
        pos = [-1] * grafo.n_nos                     # estado depois de i+1 colocações
        anotado = [False] * grafo.n_nos
        for b, c, tem_anot in registro[: i + 1]:
            pos[b], anotado[b] = c, tem_anot
        wl_parcial = sum(grade.dist(pos[u], pos[v]) for u, v in grafo.arestas if pos[u] >= 0 and pos[v] >= 0)
        titulo = f"{nome} — nó {i + 1}/{grafo.n_nos} — WL parcial = {wl_parcial}"
        if nome == "YOTT":
            titulo += "\nroxo = nó com anotação"
        desenhar(ax, grafo, grade, pos, titulo, destaque=registro[i][1], cor_no=CORES[nome])
        if nome == "YOTT":
            marcados = [grade.lc[pos[v]] for v in range(grafo.n_nos) if pos[v] >= 0 and anotado[v]]
            if marcados:
                ax.scatter([k for _, k in marcados], [r for r, _ in marcados], s=60, color="#7b3fb5", zorder=2)

    return salvar_animacao(fig, atualizar, len(registro) + pausa, f"video_{nome}", fps), r.wl


def video_sa(grafo, grade, semente=0, fps=6):
    registro = []
    r = simulated_annealing(grafo, grade, semente, registro=registro)
    fig, ax = plt.subplots(figsize=(5.5, 5.8))
    pausa = 2 * fps

    def atualizar(i):
        t, tentativas, T, wl, pos = registro[min(i, len(registro) - 1)]
        titulo = "SA — posicionamento inicial aleatório" if t == 0 else f"SA — temperatura {t}"
        desenhar(ax, grafo, grade, pos, f"{titulo}\ntentativas {tentativas:,} · T = {T:.2f} · WL = {wl}"
                 .replace(",", "."), cor_no=CORES["SA"])

    return salvar_animacao(fig, atualizar, len(registro) + pausa, "video_SA", fps), r.wl


videos = [video_construtivo("YOTO", yoto, g_ex, gr_ex), video_construtivo("YOTT", yott, g_ex, gr_ex),
          video_sa(g_ex, gr_ex)]
for caminho, wl in videos:
    print(f"{caminho}  (WL final = {wl})")
    if caminho.suffix == ".mp4":
        display(Video(str(caminho), embed=True, width=420))
    else:
        display(Image(filename=str(caminho), width=420))

## 12. Experimentos
Cada experimento grava um CSV em `resultados/`. Todo resultado é validado antes de entrar na tabela.
Sementes fixas: rodar de novo dá os mesmos WL (os tempos variam um pouco com a máquina).

| Exp. | Pergunta | Grafos |
|---|---|---|
| E1 | Qual algoritmo dá o menor WL nos DFGs reais, e a que custo de tempo? | 23 reais, 10 sementes |
| E2 | Quão longe do ótimo exato cada um fica? | 50 de ótimo conhecido |
| E3 | Com o mesmo tempo, quem chega mais perto? | 4 grafos, 3 sementes |
| E4 | Como o tempo cresce com \|V\|? | 35 sintéticos (64 a 4096 nós) |
| E5 | Quanta memória cada um usa? | 23 reais + 3 sintéticos grandes |

In [ ]:
def rodar(exp, grafo, nome_alg, funcao, semente, otimo=None, **extra):
    """Roda um algoritmo uma vez, valida o resultado e devolve uma linha da tabela."""
    grade = Grade.para_grafo(grafo.n_nos)
    r = funcao(grafo, grade, semente)
    validar(grafo, grade, r.pos)
    linha = {"exp": exp, "grafo": grafo.nome, "V": grafo.n_nos, "E": grafo.n_arestas, "alg": nome_alg,
             "semente": semente, **metricas(grafo, grade, r.pos), "otimo": otimo,
             "gap_otimo": r.wl / otimo if otimo else None, "tempo_s": r.tempo,
             **{f"n_{k}": v for k, v in r.contadores.items()}, **extra}
    return linha, r


def salvar(exp, linhas):
    df = pd.DataFrame(linhas)
    df.to_csv(SAIDA / f"{exp}.csv", index=False)
    print(f"{exp}: {len(df)} execuções → {SAIDA / (exp + '.csv')}")
    return df

### E1 — DFGs reais
23 grafos × 5 algoritmos × 10 sementes = 1150 execuções.

In [ ]:
t0 = time.perf_counter()
linhas = []
for g in REAIS:
    for nome, funcao in ALGORITMOS.items():
        for s in range(10):
            linhas.append(rodar("E1", g, nome, funcao, s)[0])
    print(f"  {g.nome:15} ({g.n_nos} nós) ok")
E1 = salvar("E1", linhas)
print(f"E1 levou {time.perf_counter() - t0:.0f} s")

### E2 — Distância exata do ótimo
5 tamanhos × 10 grafos × 5 algoritmos = 250 execuções. Cada grafo roda com a própria semente (0 a 9).

In [ ]:
t0 = time.perf_counter()
linhas = []
for lado in (6, 8, 10, 14, 20):
    for s in range(10):
        g = OTIMOS[f"optimal_{lado}x{lado}_s{s}"]
        for nome, funcao in ALGORITMOS.items():
            linhas.append(rodar("E2", g, nome, funcao, s, otimo=g.n_arestas)[0])
    print(f"  {lado}×{lado} ok")
E2 = salvar("E2", linhas)
print(f"E2 levou {time.perf_counter() - t0:.0f} s")

### E3 — Mesmo orçamento de tempo
O SA roda em 7 níveis de esforço (k movimentos por nó em cada temperatura); cada nível vira um ponto
(tempo, WL). O orçamento é 1,1 × o tempo do SA mais lento. YOTO e YOTT fazem multi-start até esgotar esse
orçamento, registrando o melhor WL após cada instância (uma curva).

In [ ]:
t0 = time.perf_counter()
GRAFOS_E3 = [TODOS["matmul"], TODOS["invert_matrix"], OTIMOS["optimal_20x20_s0"], SINTETICOS["synthetic_n512_s100"]]
ESFORCOS = [0.5, 1, 2, 5, 10, 20, 40]
linhas = []
for g in GRAFOS_E3:
    otimo = g.n_arestas if g.nome.startswith("optimal") else None
    tempos_sa = []
    for s in range(3):
        for k in ESFORCOS:
            linha, r = rodar("E3", g, "SA", lambda gg, gr, ss: simulated_annealing(gg, gr, ss, k=k), s,
                             otimo=otimo, esforco=k, tipo="ponto")
            linhas.append(linha)
            tempos_sa.append(r.tempo)
    orcamento = 1.1 * max(tempos_sa)
    for nome, alg in (("YOTO", yoto), ("YOTT", yott)):
        for s in range(3):
            linha, r = rodar("E3", g, nome, lambda gg, gr, ss: multi_start(alg, gg, gr, ss, orcamento=orcamento),
                             s, otimo=otimo, tipo="final")
            linhas.append(linha)
            # curva: só os pontos em que o melhor WL muda (e o último)
            for i, (t, wl) in enumerate(r.curva):
                if i == 0 or i == len(r.curva) - 1 or wl != r.curva[i - 1][1]:
                    linhas.append({"exp": "E3", "grafo": g.nome, "V": g.n_nos, "alg": nome, "semente": s,
                                   "tempo_s": t, "wl": wl, "lb": limitante_inferior(g), "otimo": otimo,
                                   "tipo": "curva"})
    print(f"  {g.nome:22} orçamento {orcamento:.1f} s ok")
E3 = salvar("E3", linhas)
print(f"E3 levou {time.perf_counter() - t0:.0f} s")

### E4 — Escalabilidade
7 tamanhos × 5 grafos. YOTO e YOTT em todos; SA até 2048 nós (em 4096 levaria minutos por execução).

In [ ]:
t0 = time.perf_counter()
linhas = []
for n in (64, 128, 256, 512, 1024, 2048, 4096):
    for i in range(5):
        g = SINTETICOS[f"synthetic_n{n}_s{100 + i}"]
        nomes = ["YOTO", "YOTT"] + (["SA"] if n <= 2048 else [])
        for nome in nomes:
            linhas.append(rodar("E4", g, nome, ALGORITMOS[nome], 0)[0])
    print(f"  {n} nós ok")
E4 = salvar("E4", linhas)
print(f"E4 levou {time.perf_counter() - t0:.0f} s")

### E5 — Memória
Execução separada (o `tracemalloc` deixa o código mais lento). Mede o **pico de memória em KB** durante a
execução de cada algoritmo.

In [ ]:
t0 = time.perf_counter()
linhas = []
for g in REAIS + [SINTETICOS[f"synthetic_n{n}_s100"] for n in (512, 1024, 2048)]:
    grade = Grade.para_grafo(g.n_nos)
    for nome, alg in (("YOTO", yoto), ("YOTT", yott), ("SA", simulated_annealing)):
        tracemalloc.start()
        r = alg(g, grade, 0)
        _, pico = tracemalloc.get_traced_memory()
        tracemalloc.stop()
        linhas.append({"exp": "E5", "grafo": g.nome, "V": g.n_nos, "E": g.n_arestas, "M": grade.tamanho,
                       "alg": nome, "pico_kb": pico / 1024})
E5 = salvar("E5", linhas)
print(f"E5 levou {time.perf_counter() - t0:.0f} s")

## 13. Figuras e tabelas

In [ ]:
def salvar_figura(fig, nome):
    fig.tight_layout()
    fig.savefig(FIGURAS / f"{nome}.png", dpi=160)
    plt.close(fig)


def algs_em(df):
    return [a for a in ORDEM if a in set(df["alg"])]


# ---------------- E1: tabela-resumo e duas figuras (um ponto por grafo e algoritmo)
tabela = E1.groupby("alg").agg(gap_lb_medio=("gap_lb", "mean"), gap_lb_dp=("gap_lb", "std"),
                               arestas_otimas_pct=("pct_arestas_otimas", "mean"),
                               wl_max_medio=("wl_max", "mean"), tempo_mediano_s=("tempo_s", "median"))
vencedores = E1[E1.groupby(["grafo", "semente"])["wl"].transform("min") == E1["wl"]]
tabela["vitorias_pct"] = vencedores.groupby("alg").size() / E1.groupby(["grafo", "semente"]).ngroups * 100
tabela = tabela.reindex(algs_em(E1)).fillna({"vitorias_pct": 0}).round(3)
tabela.to_csv(FIGURAS / "tabela_E1_resumo.csv")

media = E1.groupby(["grafo", "V", "alg"]).agg(gap=("gap_lb", "mean"), t=("tempo_s", "median")).reset_index()
ordem_grafos = media.drop_duplicates("grafo").sort_values("V")["grafo"].tolist()
y = {nome: i for i, nome in enumerate(ordem_grafos)}
for coluna, eixo, arquivo, log in (
        ("gap", "WL / limitante inferior (menor é melhor)", "E1_qualidade", False),
        ("t", "tempo de execução [s] (escala log)", "E1_tempo", True)):
    fig, ax = plt.subplots(figsize=(8, 0.32 * len(ordem_grafos) + 1.6))
    for i in range(len(ordem_grafos)):                  # linha-guia de cada grafo
        ax.axhline(i, color="#e1e0d9", lw=0.8, zorder=1)
    for a in algs_em(media):                           # todos os pontos de um grafo na mesma altura
        s = media[media.alg == a]
        ax.scatter(s[coluna], [y[n] for n in s.grafo], s=40, marker=MARCADORES[a], color=CORES[a],
                   edgecolor="white", linewidth=0.6, alpha=0.85, label=ROTULOS[a], zorder=3)
    ax.set_yticks(range(len(ordem_grafos)),
                  [f"{n} ({int(media[media.grafo == n].V.iloc[0])})" for n in ordem_grafos])
    ax.set_xlabel(eixo)
    if log:
        ax.set_xscale("log")
    ax.grid(axis="y", visible=False)
    ax.legend(ncol=5, loc="lower center", bbox_to_anchor=(0.5, 1.0), fontsize=9)
    ax.set_title("DFGs reais (nome e nº de nós), média de 10 sementes", pad=26, loc="left")
    salvar_figura(fig, arquivo)

# ---------------- E2: gap até o ótimo × |V|
fig, ax = plt.subplots(figsize=(7, 4.6))
for a in algs_em(E2):
    s = E2[E2.alg == a].groupby("V")["gap_otimo"].agg(["mean", "std"])
    ax.plot(s.index, s["mean"], marker=MARCADORES[a], color=CORES[a], label=ROTULOS[a])
    ax.fill_between(s.index, s["mean"] - s["std"], s["mean"] + s["std"], color=CORES[a], alpha=0.12, lw=0)
ax.axhline(1, color="#898781", lw=1, ls="--")
ax.set_xlabel("|V| (nós)")
ax.set_ylabel("WL / WL ótimo (menor é melhor)")
ax.set_title("Distância ao ótimo conhecido", loc="left")
ax.legend(loc="upper left")
salvar_figura(fig, "E2_gap_otimo")
E2.groupby(["V", "alg"])["gap_otimo"].agg(["mean", "std", "min"]).round(3).to_csv(FIGURAS / "tabela_E2_gap_otimo.csv")

# ---------------- E3: melhor WL × tempo (curvas dos construtivos, pontos do SA)
fig, axs = plt.subplots(2, 2, figsize=(11, 8))
for ax, nome_g in zip(axs.flat, E3.grafo.unique()):
    d = E3[E3.grafo == nome_g]
    ref = d["otimo"].dropna().iloc[0] if d["otimo"].notna().any() else d["lb"].iloc[0]
    eixo_t = np.logspace(np.log10(d.tempo_s[d.tempo_s > 0].min()), np.log10(d.tempo_s.max()), 200)
    for a, cor in (("YOTO", CORES["YOTO-x100"]), ("YOTT", CORES["YOTT-x100"])):
        valores = []
        for _, c in d[(d.alg == a) & (d.tipo == "curva")].groupby("semente"):
            c = c.sort_values("tempo_s")
            idx = np.searchsorted(c.tempo_s.values, eixo_t, side="right") - 1
            valores.append(np.where(idx >= 0, c.wl.values[np.clip(idx, 0, None)], np.nan))
        ax.plot(eixo_t, np.vstack(valores).mean(axis=0) / ref, color=cor, label=f"{a} multi-start")
    pontos = d[(d.alg == "SA") & (d.tipo == "ponto")].groupby("esforco")[["tempo_s", "wl"]].mean()
    ax.plot(pontos.tempo_s, pontos.wl / ref, color=CORES["SA"], marker="D", lw=1.2, label="SA (esforço crescente)")
    for k, p in pontos.iterrows():
        ax.annotate(f"{k:g}", (p.tempo_s, p.wl / ref), textcoords="offset points", xytext=(4, 4), fontsize=7)
    ax.set_xscale("log")
    ax.set_xlabel("tempo [s]")
    ax.set_ylabel("melhor WL / " + ("ótimo" if d["otimo"].notna().any() else "LB"))
    ax.set_title(f"{nome_g} ({int(d.V.iloc[0])} nós)", loc="left")
    ax.legend(fontsize=8)
fig.suptitle("Qualidade × tempo com o mesmo orçamento (rótulos do SA = k)", x=0.01, ha="left")
salvar_figura(fig, "E3_qualidade_x_tempo")
# tabela: melhor WL no fim do orçamento (YOTO/YOTT) e com o maior esforço (SA, k = 40); média das 3 sementes
finais = E3[(E3.tipo == "final") | ((E3.tipo == "ponto") & (E3.esforco == max(ESFORCOS)))]
finais.pivot_table(index="grafo", columns="alg", values="wl", aggfunc="mean").round(1) \
    .to_csv(FIGURAS / "tabela_E3_melhor_wl.csv")

# ---------------- E4: tempo × |V| em log-log e inclinações
fig, ax = plt.subplots(figsize=(7, 4.6))
inclinacoes = []
for a in algs_em(E4):
    s = E4[E4.alg == a].groupby("V")["tempo_s"].median()
    b = np.polyfit(np.log10(s.index), np.log10(s.values), 1)[0]
    ax.plot(s.index, s.values, marker=MARCADORES[a], color=CORES[a], label=f"{ROTULOS[a]} (inclinação ≈ {b:.2f})")
    inclinacoes.append({"alg": a, "inclinacao": round(b, 3), "dobrar_V_multiplica_tempo_por": round(2 ** b, 2)})
ax.set_xscale("log", base=2)
ax.set_yscale("log")
ax.set_xlabel("|V| (nós)")
ax.set_ylabel("tempo mediano [s]")
ax.set_title("Escalabilidade: tempo × |V| (log-log)", loc="left")
ax.legend(loc="upper left")
salvar_figura(fig, "E4_escalabilidade")
pd.DataFrame(inclinacoes).to_csv(FIGURAS / "tabela_E4_inclinacoes.csv", index=False)
colunas = [c for c in ("n_bfs", "n_temperaturas", "n_tentativas") if c in E4]
E4.groupby(["alg", "V"])[colunas].mean().round(1).to_csv(FIGURAS / "tabela_E4_contadores.csv")

# ---------------- E5: pico de memória × |V| (log-log; inclinação ≈ 1 = espaço linear)
fig, ax = plt.subplots(figsize=(7, 4.6))
resumo_mem = []
for a in algs_em(E5):
    kb = E5[E5.alg == a].groupby("V")["pico_kb"].mean()
    ax.plot(kb.index, kb.values, marker=MARCADORES[a], color=CORES[a], label=ROTULOS[a])
    grandes = kb[kb.index >= 512]
    resumo_mem.append({"alg": a, "inclinacao": round(np.polyfit(np.log(kb.index), np.log(kb.values), 1)[0], 3),
                       "kb_por_no_V>=512": round((grandes / grandes.index).mean(), 3)})
ax.set_xscale("log", base=2)
ax.set_yscale("log")
ax.set_xlabel("|V| (nós)")
ax.set_ylabel("pico de memória [KB]")
ax.set_title("Memória medida (pico)", loc="left")
ax.legend(loc="upper left")
salvar_figura(fig, "E5_memoria")
pd.DataFrame(resumo_mem).to_csv(FIGURAS / "tabela_E5_memoria.csv", index=False)
print("figuras e tabelas em", FIGURAS)

## 14. Mostrar e baixar os resultados

In [ ]:
for csv in sorted(FIGURAS.glob("tabela_*.csv")):
    display(Markdown(f"**{csv.stem}**"))
    display(pd.read_csv(csv))
for png in sorted(FIGURAS.glob("E*.png")):
    display(Markdown(f"**{png.stem}**"))
    display(Image(filename=str(png), width=760))

pacote = Path("resultados.zip")
with zipfile.ZipFile(pacote, "w", zipfile.ZIP_DEFLATED) as z:
    for arquivo in SAIDA.rglob("*"):
        if arquivo.is_file():
            z.write(arquivo)
print("Arquivo gerado:", pacote.resolve())
try:
    from google.colab import files
    files.download(str(pacote))
except ImportError:
    pass   # fora do Colab, o .zip fica na pasta atual